# Tournament Simulation

In this notebook, we use the trained Poisson models to simulate the full 2026 World Cup.

The goal is to:

1. Load the trained home-goals and away-goals models
2. Build match-level predictions for any two teams
3. Simulate group-stage matches
4. Rank teams inside each group
5. Simulate the knockout bracket
6. Run the tournament many times using Monte Carlo simulation
7. Estimate each team's probability of reaching each stage

In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

# Section 1: Load models and data

In [2]:
# Project paths
DATA_DIR = Path("../data")
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REFERENCE_DIR = DATA_DIR / "reference"
MODELS_DIR = Path("../models")

In [3]:
# Load trained Poisson models
with open(MODELS_DIR / "poisson_home.pkl", "rb") as f:
    model_home = pickle.load(f)

with open(MODELS_DIR / "poisson_away.pkl", "rb") as f:
    model_away = pickle.load(f)

# Load feature column order used during training
with open(MODELS_DIR / "feature_columns.pkl", "rb") as f:
    feature_columns = pickle.load(f)

print("Models loaded successfully.")
print(f"Number of model features: {len(feature_columns)}")

Models loaded successfully.
Number of model features: 17


In [4]:
# Core datasets
df_historical = pd.read_csv(INTERIM_DIR / "historical_matches.csv")
df_fixtures = pd.read_csv(INTERIM_DIR / "wc2026_fixtures.csv")

# Processed feature datasets
df_match_features = pd.read_csv(PROCESSED_DIR / "df_match_features.csv")
df_form = pd.read_csv(PROCESSED_DIR / "df_form_2026.csv")
df_h2h = pd.read_csv(PROCESSED_DIR / "df_h2h_2026.csv")

# Reference datasets
df_confederations = pd.read_csv(REFERENCE_DIR / "FIFA_confederations.csv")
df_knockout = pd.read_csv(REFERENCE_DIR / "fixtures_knockout_wc2026.csv")
df_groups = pd.read_csv(REFERENCE_DIR / "group_stages.csv", sep=";")

# FIFA ranking reference
df_fifa_rank = pd.read_csv(DATA_DIR / "raw" / "wc_2026_48_teams_fifa_rank_change_corrected.csv")

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [5]:
# Inspect shape
datasets = {
    "historical_matches": df_historical,
    "wc2026_fixtures": df_fixtures,
    "df_match_features": df_match_features,
    "df_form_2026": df_form,
    "df_h2h_2026": df_h2h,
    "FIFA_confederations": df_confederations,
    "group_stages": df_groups,
    "fixtures_knockout_wc2026": df_knockout,
    "fifa_rank_2026": df_fifa_rank,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

historical_matches: (49215, 9)
wc2026_fixtures: (72, 9)
df_match_features: (49048, 16)
df_form_2026: (48, 2)
df_h2h_2026: (565, 4)
FIFA_confederations: (48, 2)
group_stages: (48, 3)
fixtures_knockout_wc2026: (32, 8)
fifa_rank_2026: (48, 4)


In [6]:
# Inspect columns
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.columns.tolist())


historical_matches
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']

wc2026_fixtures
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']

df_match_features
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'home_elo_pre', 'away_elo_pre', 'elo_diff', 'tournament_weight', 'is_competitive', 'home_confederation', 'away_confederation']

df_form_2026
['team', 'form_score']

df_h2h_2026
['team_a', 'team_b', 'h2h_score', 'n_meetings_analysed']

FIFA_confederations
['nation', 'confederation']

group_stages
['group', 'position', 'nation']

fixtures_knockout_wc2026
['match_id', 'round', 'match_date', 'match_time', 'home_slot', 'away_slot', 'winner_advances_to', 'loser_advances_to']

fifa_rank_2026
['Nation', 'FIFA_2022_ranking', 'FIFA_2026_rank', 'rank_change']


In [7]:
# Check number of World Cup teams
teams_from_groups = set(df_groups["nation"])
teams_from_confederations = set(df_confederations["nation"])
teams_from_form = set(df_form["team"])
teams_from_fifa = set(df_fifa_rank["Nation"])

print("Teams in group_stages:", len(teams_from_groups))
print("Teams in confederations:", len(teams_from_confederations))
print("Teams in form table:", len(teams_from_form))
print("Teams in FIFA ranking table:", len(teams_from_fifa))

# Check missing teams across key reference tables
print("\nTeams in groups but missing from confederations:")
print(sorted(teams_from_groups - teams_from_confederations))

print("\nTeams in groups but missing from form table:")
print(sorted(teams_from_groups - teams_from_form))

print("\nTeams in groups but missing from FIFA ranking table:")
print(sorted(teams_from_groups - teams_from_fifa))

Teams in group_stages: 48
Teams in confederations: 48
Teams in form table: 48
Teams in FIFA ranking table: 48

Teams in groups but missing from confederations:
[]

Teams in groups but missing from form table:
[]

Teams in groups but missing from FIFA ranking table:
[]


In [8]:
display(df_fixtures.head())
display(df_groups.head())
display(df_knockout.head())
display(df_match_features.head())

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,2026-06-11,Mexico,South Africa,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False
1,2026-06-11,South Korea,Czech Republic,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True
2,2026-06-12,Canada,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Toronto,Canada,False
3,2026-06-12,United States,Paraguay,NaN,NaN,FIFA World Cup,Inglewood,United States,False
4,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True


,group,position,nation
0,A,1,Mexico
1,A,2,South Africa
2,A,3,South Korea
3,A,4,Czech Republic
4,B,1,Canada


,match_id,round,match_date,match_time,home_slot,away_slot,winner_advances_to,loser_advances_to
0,M73,R32,2026-06-28,20:00,2A,2B,M90,NaN
1,M74,R32,2026-06-29,21:30,1E,3ABCDF,M89,NaN
2,M75,R32,2026-06-30,02:00,1F,2C,M90,NaN
3,M76,R32,2026-06-29,18:00,1C,2F,M91,NaN
4,M77,R32,2026-06-30,22:00,1I,3CDFGH,M89,NaN


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo_pre,away_elo_pre,elo_diff,tournament_weight,is_competitive,home_confederation,away_confederation
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.000000,1500.000000,0.000000,1,False,UEFA,UEFA
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1502.801300,1497.198700,5.602600,1,False,UEFA,UEFA
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1486.622531,1513.377469,-26.754938,1,False,UEFA,UEFA
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1505.454946,1494.545054,10.909891,1,False,UEFA,UEFA
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1497.633108,1502.366892,-4.733783,1,False,UEFA,UEFA


#
----

# Section 2: create lookup tables for Elo, form, H2H, and confederation.

The aim here is to make it easy to fetch, for any team:
- Elo rating
- confederation
- recent form score
- FIFA rank
- H2H score against another team

## 2.1— Create lookup tables

Before we can predict any 2026 match, we need quick lookup tables for team-level and matchup-level information.

In this section, we create:

- latest Elo rating per team
- team-to-confederation mapping
- team-to-form-score mapping
- team-to-FIFA-rank mapping
- head-to-head lookup between two teams

These lookup tables will make the prediction and simulation functions much cleaner.

In [9]:
#### prepare latest Elo per team

# Make sure dates are treated as dates
df_match_features["date"] = pd.to_datetime(df_match_features["date"])

# Home-team Elo records
home_elo = df_match_features[["date", "home_team", "home_elo_pre"]].rename(
    columns={
        "home_team": "team",
        "home_elo_pre": "elo"
    }
)

# Away-team Elo records
away_elo = df_match_features[["date", "away_team", "away_elo_pre"]].rename(
    columns={
        "away_team": "team",
        "away_elo_pre": "elo"
    }
)

# Combine home and away Elo records
df_team_elo = pd.concat([home_elo, away_elo], ignore_index=True)

# Get latest available Elo before the 2026 tournament
df_latest_elo = (
    df_team_elo
    .sort_values("date")
    .drop_duplicates(subset="team", keep="last")
    .reset_index(drop=True)
)

team_to_elo = dict(zip(df_latest_elo["team"], df_latest_elo["elo"]))

df_latest_elo.tail()

,date,team,elo
317,2026-03-31,Republic of Ireland,1748.177082
318,2026-03-31,Russia,1840.274201
319,2026-03-31,Spain,2229.756779
320,2026-03-31,Curaçao,1621.162116
321,2026-03-31,Comoros,1488.747627


In [10]:
df_latest_elo.sort_values("elo", ascending=False).head(13)

,date,team,elo
319,2026-03-31,Spain,2229.756779
295,2026-03-31,Argentina,2179.651451
197,2026-03-29,France,2124.778622
296,2026-03-31,England,2094.723009
207,2026-03-29,Colombia,2064.025447
292,2026-03-31,Brazil,2050.192145
248,2026-03-31,Portugal,2023.470091
298,2026-03-31,Netherlands,2019.419663
270,2026-03-31,Ecuador,2015.364338
255,2026-03-31,Croatia,1994.757090


In [11]:
#### create confederation lookup

team_to_confederation = dict(
    zip(df_confederations["nation"], df_confederations["confederation"])
)

list(team_to_confederation.items())[:5]

[('France', 'UEFA'),
 ('Spain', 'UEFA'),
 ('England', 'UEFA'),
 ('Portugal', 'UEFA'),
 ('Netherlands', 'UEFA')]

In [12]:
#### create form lookup (last 10 games goal differential)

team_to_form = dict(
    zip(df_form["team"], df_form["form_score"])
)

list(team_to_form.items())[:5]

[('Morocco', 63.0),
 ('Japan', 61.0),
 ('Spain', 59.0),
 ('Portugal', 56.0),
 ('England', 54.0)]

In [13]:
#### create FIFA ranking lookup as of 2026

team_to_fifa_rank = dict(
    zip(df_fifa_rank["Nation"], df_fifa_rank["FIFA_2026_rank"])
)

team_to_fifa_rank_change = dict(
    zip(df_fifa_rank["Nation"], df_fifa_rank["rank_change"])
)

list(team_to_fifa_rank.items())[:8]

[('France', 1),
 ('Spain', 2),
 ('Argentina', 3),
 ('England', 4),
 ('Portugal', 5),
 ('Brazil', 6),
 ('Netherlands', 7),
 ('Morocco', 8)]

In [14]:
#### create H2H helper function

def get_h2h_score(team_a, team_b):
    """
    Return the head-to-head score from team_a's perspective.

    Example:
    If Argentina vs Brazil = +6,
    then Brazil vs Argentina = -6.
    """

    direct_match = df_h2h[
        (df_h2h["team_a"] == team_a) &
        (df_h2h["team_b"] == team_b)
    ]

    if len(direct_match) > 0:
        return direct_match["h2h_score"].iloc[0]

    reverse_match = df_h2h[
        (df_h2h["team_a"] == team_b) &
        (df_h2h["team_b"] == team_a)
    ]

    if len(reverse_match) > 0:
        return -reverse_match["h2h_score"].iloc[0]

    return 0

In [15]:
#### test H2H helper (team head to head, only looks at last 5 games but gives 0 for teams that only played 1 or 2 games)

test_pairs = [
    ("Argentina", "Brazil"),
    ("Brazil", "Argentina"),
    ("France", "Spain"),
    ("Spain", "France"),
    ("France", "Senegal"),
    ("Algeria", "Morocco")
]

for team_a, team_b in test_pairs:
    print(f"{team_a} vs {team_b}: {get_h2h_score(team_a, team_b)}")

Argentina vs Brazil: 6.0
Brazil vs Argentina: -6.0
France vs Spain: -2.0
Spain vs France: 2.0
France vs Senegal: 0.0
Algeria vs Morocco: -6.0


In [16]:
#### sanity check all World Cup teams have lookup values

wc_teams = sorted(df_groups["nation"].unique())

missing_elo = [team for team in wc_teams if team not in team_to_elo]
missing_confederation = [team for team in wc_teams if team not in team_to_confederation]
missing_form = [team for team in wc_teams if team not in team_to_form]
missing_fifa_rank = [team for team in wc_teams if team not in team_to_fifa_rank]

print("Missing Elo:", missing_elo)
print("Missing confederation:", missing_confederation)
print("Missing form:", missing_form)
print("Missing FIFA rank:", missing_fifa_rank)

Missing Elo: []
Missing confederation: []
Missing form: []
Missing FIFA rank: []


In [17]:
#### inspect one team profile

sample_team = "Morocco"

team_profile = {
    "team": sample_team,
    "elo": team_to_elo.get(sample_team),
    "confederation": team_to_confederation.get(sample_team),
    "form_score": team_to_form.get(sample_team),
    "fifa_rank": team_to_fifa_rank.get(sample_team),
    "rank_change": team_to_fifa_rank_change.get(sample_team),
}

team_profile

{'team': 'Morocco',
 'elo': 1972.941440105271,
 'confederation': 'CAF',
 'form_score': 63.0,
 'fifa_rank': 8,
 'rank_change': 3}

#### Choice:
- Keep the prediction model pure. Use only the features it was trained and validated on. 
- Use FIFA rank, form, and H2H only for dashboard context and explanations.

#### Use the model for probabilities
Poisson model:
- Elo + match context → expected goals → win/draw/loss probabilities

#### Use the extra football context for interpretation:
Dashboard explanation:
- FIFA rank, rank change, recent form, H2H, confederation, Elo difference

PS: I also calculated recent form, FIFA rankings, and head-to-head records. But I did not feed them into the model because they were not part of the validated training setup. Instead, I use them in the dashboard to explain each matchup alongside the model probabilities

#
----

# Section 3

## 3.1 — Build match prediction function

Now that the lookup tables are ready, we can create a prediction function for any 2026 matchup.

The model only uses the features it was trained on:

- home Elo
- away Elo
- Elo difference
- tournament weight
- neutral venue
- home confederation
- away confederation

The function will:

1. Build a one-row feature table
2. One-hot encode categorical columns
3. Align the columns with `feature_columns.pkl`
4. Predict expected goals using the two Poisson models
5. Convert expected goals into win/draw/loss probabilities

## Build Model feature row

In [18]:
def build_match_features(home_team, away_team, neutral=True, tournament_weight=5):
    """
    Build one prediction row for a 2026 World Cup match.

    This function only includes the features used during model training.
    """

    home_elo = team_to_elo.get(home_team)
    away_elo = team_to_elo.get(away_team)

    home_conf = team_to_confederation.get(home_team)
    away_conf = team_to_confederation.get(away_team)

    if home_elo is None:
        raise ValueError(f"Missing Elo rating for {home_team}")

    if away_elo is None:
        raise ValueError(f"Missing Elo rating for {away_team}")

    if home_conf is None:
        raise ValueError(f"Missing confederation for {home_team}")

    if away_conf is None:
        raise ValueError(f"Missing confederation for {away_team}")

    row = pd.DataFrame([{
        "home_elo_pre": home_elo,
        "away_elo_pre": away_elo,
        "elo_diff": home_elo - away_elo,
        "tournament_weight": tournament_weight,
        "neutral": int(neutral),
        "home_confederation": home_conf,
        "away_confederation": away_conf
    }])

    # One-hot encode categorical variables
    X = pd.get_dummies(row)

    # Align with training columns
    X = X.reindex(columns=feature_columns, fill_value=0)

    # Important: statsmodels intercept column must be 1
    if "const" in X.columns:
        X["const"] = 1

    # Make sure all model inputs are numeric
    X = X.astype(float)

    return X

- It creates the exact one-row input table that the trained Poisson model expects.

- It is not predicting yet. It is only preparing the features.

In [19]:
#### test feature builder

X_test = build_match_features("Argentina", "Brazil")

print(X_test.shape)
display(X_test)

(1, 17)


,const,home_elo_pre,away_elo_pre,tournament_weight,neutral,home_confederation_CAF,home_confederation_CONCACAF,home_confederation_CONMEBOL,home_confederation_OFC,home_confederation_UEFA,home_confederation_Unknown,away_confederation_CAF,away_confederation_CONCACAF,away_confederation_CONMEBOL,away_confederation_OFC,away_confederation_UEFA,away_confederation_Unknown
0,1.0,2179.651451,2050.192145,5.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


## create Poisson probability function

In [20]:
from scipy.stats import poisson

In [21]:
def predict_match(home_team, away_team, neutral=True, tournament_weight=5, max_goals=10):
    """
    Predict expected goals and win/draw/loss probabilities for one match.
    """

    X = build_match_features(
        home_team=home_team,
        away_team=away_team,
        neutral=neutral,
        tournament_weight=tournament_weight
    )

    home_xg = model_home.predict(X)[0]
    away_xg = model_away.predict(X)[0]

    score_probs = []

    home_win_prob = 0
    draw_prob = 0
    away_win_prob = 0

    for home_goals in range(max_goals + 1):
        for away_goals in range(max_goals + 1):

            prob = (
                poisson.pmf(home_goals, home_xg) *
                poisson.pmf(away_goals, away_xg)
            )

            score_probs.append({
                "home_goals": home_goals,
                "away_goals": away_goals,
                "probability": prob
            })

            if home_goals > away_goals:
                home_win_prob += prob
            elif home_goals == away_goals:
                draw_prob += prob
            else:
                away_win_prob += prob

    score_probs = pd.DataFrame(score_probs)

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_xg": home_xg,
        "away_xg": away_xg,
        "home_win_prob": home_win_prob,
        "draw_prob": draw_prob,
        "away_win_prob": away_win_prob,
        "score_probs": score_probs
    }

## Test Predictions

In [22]:
test_matches = [
    ("Argentina", "Brazil"),
    ("France", "Spain"),
    ("Morocco", "Portugal"),
    ("England", "Iran"),
    ("Mexico", "South Africa")
]

for home_team, away_team in test_matches:
    pred = predict_match(home_team, away_team)

    print(f"\n{home_team} vs {away_team}")
    print(f"Home xG: {pred['home_xg']:.2f}")
    print(f"Away xG: {pred['away_xg']:.2f}")
    print(f"Home win: {pred['home_win_prob']:.1%}")
    print(f"Draw: {pred['draw_prob']:.1%}")
    print(f"Away win: {pred['away_win_prob']:.1%}")


Argentina vs Brazil
Home xG: 1.36
Away xG: 0.82
Home win: 49.4%
Draw: 27.9%
Away win: 22.7%

France vs Spain
Home xG: 0.77
Away xG: 1.48
Home win: 19.4%
Draw: 26.4%
Away win: 54.2%

Morocco vs Portugal
Home xG: 0.93
Away xG: 1.04
Home win: 31.4%
Draw: 31.1%
Away win: 37.5%

England vs Iran
Home xG: 1.26
Away xG: 0.83
Home win: 46.3%
Draw: 29.1%
Away win: 24.6%

Mexico vs South Africa
Home xG: 2.11
Away xG: 0.48
Home win: 75.7%
Draw: 17.2%
Away win: 7.1%


In [23]:
df_latest_elo.sort_values("elo", ascending=False).head(3)

,date,team,elo
319,2026-03-31,Spain,2229.756779
295,2026-03-31,Argentina,2179.651451
197,2026-03-29,France,2124.778622


In [24]:
#### test H2H helper (team head to head, only looks at last 5 games but gives 0 for teams that only played 1 or 2 games)

test_pairs = [
    ("France", "Spain"),
    ("Portugal", "Morocco")
]

for team_a, team_b in test_pairs:
    print(f"{team_a} vs {team_b}: {get_h2h_score(team_a, team_b)}")

France vs Spain: -2.0
Portugal vs Morocco: -2.0


## inspect most likely scorelines

In [27]:
pred = predict_match("Argentina", "Brazil")

pred["score_probs"].sort_values(
    "probability",
    ascending=False
).head(10)

,home_goals,away_goals,probability
11,1,0,0.153658
12,1,1,0.126244
0,0,0,0.113325
22,2,0,0.104173
1,0,1,0.093107
23,2,1,0.085587
13,1,2,0.051860
33,3,0,0.047083
34,3,1,0.038683
2,0,2,0.038248


In [29]:
pred = predict_match("Portugal", "Morocco")

pred["score_probs"].sort_values(
    "probability",
    ascending=False
).head(10)

,home_goals,away_goals,probability
11,1,0,0.135903
12,1,1,0.135311
0,0,0,0.133433
1,0,1,0.132852
22,2,0,0.069209
23,2,1,0.068908
13,1,2,0.067361
2,0,2,0.066137
24,2,2,0.034304
33,3,0,0.023497


In [28]:
pred["home_win_prob"] + pred["draw_prob"] + pred["away_win_prob"]

np.float64(0.9999997916000778)

- The win/draw/loss probabilities sum to almost 1.
- The most likely scorelines are realistic for the predicted expected goals.
- This suggests the prediction function is working as expected.

#
-----

# Section 4 — simulate a single match.

## 4.1 — Simulate a single match

Now that we can predict expected goals for any matchup, we can simulate one match result.

The logic is:

1. Use the Poisson model to predict expected goals for each team
2. Sample actual goals from each team's Poisson distribution
3. Return the score and match result

For group-stage matches, draws are allowed.
Knockout matches will need extra logic later because one team must advance.

In [30]:
def simulate_match(home_team, away_team, neutral=True, tournament_weight=5, random_state=None):
    """
    Simulate one football match using the predicted expected goals.

    For group-stage matches:
    - home win is allowed
    - draw is allowed
    - away win is allowed
    """

    if random_state is not None:
        np.random.seed(random_state)

    pred = predict_match(
        home_team=home_team,
        away_team=away_team,
        neutral=neutral,
        tournament_weight=tournament_weight
    )

    home_xg = pred["home_xg"]
    away_xg = pred["away_xg"]

    home_goals = np.random.poisson(home_xg)
    away_goals = np.random.poisson(away_xg)

    if home_goals > away_goals:
        result = "H"
        winner = home_team
    elif home_goals < away_goals:
        result = "A"
        winner = away_team
    else:
        result = "D"
        winner = None

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_xg": home_xg,
        "away_xg": away_xg,
        "home_goals": home_goals,
        "away_goals": away_goals,
        "result": result,
        "winner": winner
    }

In [34]:
#### Test one match

simulate_match("Argentina", "Brazil", random_state=16)

{'home_team': 'Argentina',
 'away_team': 'Brazil',
 'home_xg': np.float64(1.3559041758011838),
 'away_xg': np.float64(0.8215896270684804),
 'home_goals': 0,
 'away_goals': 1,
 'result': 'A',
 'winner': 'Brazil'}

In [35]:
simulate_match("Argentina", "Brazil", random_state=42)

{'home_team': 'Argentina',
 'away_team': 'Brazil',
 'home_xg': np.float64(1.3559041758011838),
 'away_xg': np.float64(0.8215896270684804),
 'home_goals': 3,
 'away_goals': 0,
 'result': 'H',
 'winner': 'Argentina'}

- One simulated match can look surprising. That is normal.

- The whole point of Monte Carlo is that we do not trust one run. We run the tournament thousands of times, then count the patterns.

## Simulate the same match multiple times

In [36]:
for i in range(10):
    match = simulate_match("Argentina", "Brazil")

    print(
        f"{match['home_team']} {match['home_goals']} - "
        f"{match['away_goals']} {match['away_team']} | "
        f"Result: {match['result']}"
    )

Argentina 0 - 0 Brazil | Result: D
Argentina 3 - 2 Brazil | Result: H
Argentina 0 - 0 Brazil | Result: D
Argentina 1 - 0 Brazil | Result: H
Argentina 1 - 0 Brazil | Result: H
Argentina 1 - 1 Brazil | Result: D
Argentina 0 - 1 Brazil | Result: A
Argentina 0 - 1 Brazil | Result: A
Argentina 0 - 3 Brazil | Result: A
Argentina 0 - 1 Brazil | Result: A


## Test a few different match-ups

In [39]:
sample_matches = [
    ("Argentina", "Brazil"),
    ("France", "Spain"),
    ("Morocco", "Portugal"),
    ("England", "Iran"),
    ("Mexico", "South Africa")
]

for home_team, away_team in sample_matches:
    match = simulate_match(home_team, away_team)

    print(
        f"{match['home_team']} {match['home_goals']} - "
        f"{match['away_goals']} {match['away_team']} "
        f"| xG: {match['home_xg']:.2f} - {match['away_xg']:.2f} "
        f"| Result: {match['result']}"
    )

Argentina 0 - 2 Brazil | xG: 1.36 - 0.82 | Result: A
France 0 - 1 Spain | xG: 0.77 - 1.48 | Result: A
Morocco 3 - 0 Portugal | xG: 0.93 - 1.04 | Result: H
England 3 - 1 Iran | xG: 1.26 - 0.83 | Result: H
Mexico 1 - 0 South Africa | xG: 2.11 - 0.48 | Result: H


In [64]:
### Simulate 10,000 times to see if averages make sense

results = []

for i in range(10_000):
    match = simulate_match("Argentina", "Brazil")
    results.append(match["result"])

pd.Series(results).value_counts(normalize=True).sort_index()

A    0.2332
D    0.2747
H    0.4921
Name: proportion, dtype: float64

The single-match outputs will look noisy. That is the whole point of simulation. What matters is that thousands of simulated matches average out to the model probabilities.

So now we have confirmed three things:

1. predict_match() gives sensible xG and probabilities.
2. simulate_match() creates random match outcomes correctly.
3. Repeating the same match many times converges back to the model probabilities.

- ##### One small note: 10_000 took 42 seconds, which is slow because each simulation calls predict_match() again. Later, for the full tournament, we should avoid recalculating xG every time where possible.

# 
-----

# Section 5 — simulate the group stage.

## 5.1 — Simulate the group stage

Now we simulate the 72 group-stage matches.

For each match, we will:

1. Simulate the score using `simulate_match()`
2. Award points
3. Update goals for, goals against, and goal difference
4. Build one table per group
5. Rank teams using points, goal difference, and goals scored

For this first version, if teams are still tied after those rules, we use a random tie-breaker.

## add group information to fixtures

In [43]:
# Team to group lookup
team_to_group = dict(zip(df_groups["nation"], df_groups["group"]))

# Add group to each group-stage fixture
df_group_fixtures = df_fixtures.copy()

df_group_fixtures["group"] = df_group_fixtures["home_team"].map(team_to_group)

# Basic check
display(df_group_fixtures.head())
print(df_group_fixtures["group"].value_counts().sort_index())

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,group
0,2026-06-11,Mexico,South Africa,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False,A
1,2026-06-11,South Korea,Czech Republic,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True,A
2,2026-06-12,Canada,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Toronto,Canada,False,B
3,2026-06-12,United States,Paraguay,NaN,NaN,FIFA World Cup,Inglewood,United States,False,D
4,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True,B


group
A    6
B    6
C    6
D    6
E    6
F    6
G    6
H    6
I    6
J    6
K    6
L    6
Name: count, dtype: int64


Why 6 games per group:
- Team 1 vs Team 2
- Team 1 vs Team 3
- Team 1 vs Team 4
- Team 2 vs Team 3
- Team 2 vs Team 4
- Team 3 vs Team 4

In [45]:
#### check every fixture has a group

missing_group_fixtures = df_group_fixtures[df_group_fixtures["group"].isna()]

print("Fixtures with missing group:", len(missing_group_fixtures))
display(missing_group_fixtures)

Fixtures with missing group: 0


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,group


## create empty group table

In [47]:
#### create empty group table

def create_empty_group_table(group_teams):
    """
    Create an empty standings table for one group.
    """

    table = pd.DataFrame({
        "team": group_teams,
        "played": 0,
        "wins": 0,
        "draws": 0,
        "losses": 0,
        "goals_for": 0,
        "goals_against": 0,
        "goal_difference": 0,
        "points": 0
    })

    return table

## Update table after 1 Match

In [48]:
def update_group_table(table, match):
    """
    Update a group table after one simulated match.
    """

    home_team = match["home_team"]
    away_team = match["away_team"]
    home_goals = match["home_goals"]
    away_goals = match["away_goals"]

    # Update played
    table.loc[table["team"] == home_team, "played"] += 1
    table.loc[table["team"] == away_team, "played"] += 1

    # Update goals
    table.loc[table["team"] == home_team, "goals_for"] += home_goals
    table.loc[table["team"] == home_team, "goals_against"] += away_goals

    table.loc[table["team"] == away_team, "goals_for"] += away_goals
    table.loc[table["team"] == away_team, "goals_against"] += home_goals

    # Update result stats and points
    if home_goals > away_goals:
        table.loc[table["team"] == home_team, "wins"] += 1
        table.loc[table["team"] == away_team, "losses"] += 1

        table.loc[table["team"] == home_team, "points"] += 3

    elif home_goals < away_goals:
        table.loc[table["team"] == away_team, "wins"] += 1
        table.loc[table["team"] == home_team, "losses"] += 1

        table.loc[table["team"] == away_team, "points"] += 3

    else:
        table.loc[table["team"] == home_team, "draws"] += 1
        table.loc[table["team"] == away_team, "draws"] += 1

        table.loc[table["team"] == home_team, "points"] += 1
        table.loc[table["team"] == away_team, "points"] += 1

    # Recalculate goal difference
    table["goal_difference"] = table["goals_for"] - table["goals_against"]

    return table

## rank one group table

In [50]:
def rank_group_table(table):
    """
    Rank teams in a group.

    Main rules:
    1. Points
    2. Goal difference
    3. Goals scored
    4. Random tie-breaker
    """

    table = table.copy()

    # Temporary random tie-breaker for exact ties
    table["random_tiebreaker"] = np.random.random(len(table))

    table = (
        table
        .sort_values(
            by=["points", "goal_difference", "goals_for", "random_tiebreaker"],
            ascending=[False, False, False, False]
        )
        .reset_index(drop=True)
    )

    table["group_rank"] = table.index + 1

    table = table.drop(columns=["random_tiebreaker"])

    return table

## simulate one full group

In [51]:
def simulate_group(group_name):
    """
    Simulate all matches in one group and return the ranked group table.
    """

    group_teams = (
        df_groups[df_groups["group"] == group_name]
        .sort_values("position")["nation"]
        .tolist()
    )

    group_matches = df_group_fixtures[df_group_fixtures["group"] == group_name]

    table = create_empty_group_table(group_teams)

    simulated_matches = []

    for _, row in group_matches.iterrows():
        match = simulate_match(
            home_team=row["home_team"],
            away_team=row["away_team"],
            neutral=row["neutral"]
        )

        simulated_matches.append(match)
        table = update_group_table(table, match)

    ranked_table = rank_group_table(table)
    ranked_table["group"] = group_name

    return ranked_table, pd.DataFrame(simulated_matches)

## test one group

In [75]:
group_a_table, group_a_matches = simulate_group("C")

display(group_a_matches)
display(group_a_table)

,home_team,away_team,home_xg,away_xg,home_goals,away_goals,result,winner
0,Brazil,Morocco,1.196691,0.777093,0,1,A,Morocco
1,Haiti,Scotland,1.058296,1.283642,2,1,H,Haiti
2,Scotland,Morocco,0.743416,1.489087,0,2,A,Morocco
3,Brazil,Haiti,2.245960,0.589181,1,1,D,NaN
4,Scotland,Brazil,0.681838,2.062838,0,1,A,Brazil
5,Morocco,Haiti,1.766992,0.635292,0,1,A,Haiti


,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,group_rank,group
0,Haiti,3,2,1,0,4,2,2,7,1,C
1,Morocco,3,2,0,1,3,1,2,6,2,C
2,Brazil,3,1,1,1,2,2,0,4,3,C
3,Scotland,3,0,0,3,1,5,-4,0,4,C


In [73]:
group_c_rank_results = []

for i in range(1000):
    group_table, group_matches = simulate_group("C")

    group_c_rank_results.append(
        group_table[["team", "group_rank"]]
    )

df_group_c_ranks = pd.concat(group_c_rank_results, ignore_index=True)

group_c_rank_probs = (
    df_group_c_ranks
    .groupby(["team", "group_rank"])
    .size()
    .reset_index(name="count")
)

group_c_rank_probs["probability"] = group_c_rank_probs["count"] / 1000

group_c_rank_probs_pivot = (
    group_c_rank_probs
    .pivot(index="team", columns="group_rank", values="probability")
    .fillna(0)
)

group_c_rank_probs_pivot.columns = [
    f"rank_{int(col)}_prob" for col in group_c_rank_probs_pivot.columns
]

group_c_rank_probs_pivot.sort_values("rank_1_prob", ascending=False)

,rank_1_prob,rank_2_prob,rank_3_prob,rank_4_prob
team,,,,
Brazil,0.600,0.264,0.104,0.032
Morocco,0.283,0.419,0.207,0.091
Scotland,0.084,0.193,0.377,0.346
Haiti,0.033,0.124,0.312,0.531
